# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates loading and exploration of the FAIR² tabular dataset describing second primary colorectal cancer (CRC) in cancer survivors, using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described via a [Croissant schema](https://mlcommons.org/croissant/) available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed (uncomment if running for the first time)
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset metadata and explore schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Get top-level metadata
metadata = dataset.metadata

# Display basic metadata information
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")
print(f"Number of authors: {len(metadata.author) if hasattr(metadata, 'author') else 0}")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets (tables), including their `@id`, fields, and field `@id`s.

**Note:** All references to record sets and fields use their unique `@id` as required by Croissant and this notebook's template.

In [ ]:
# List all record sets by @id and by name with their fields
record_sets = [rs for rs in dataset.record_sets]
print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"   Name: {getattr(rs, 'name', '<no name>')}")
    if hasattr(rs, 'fields'):
        print("   Fields:")
        for fld in rs.fields:
            print(f"    - {fld.id} ({getattr(fld, 'name', '<no name>')}, dtype={getattr(fld, 'data_type', '<no dtype>')})")
    print()

if len(record_sets) > 0:
    print("Preview (first 2 records from first record set):\n")
    first_rs_id = record_sets[0].id
    # Display a preview using the record's @id
    for i, rec in enumerate(dataset.records(record_set=first_rs_id)):
        print(rec)
        if i >= 1:
            break

## 3. Data Extraction
Load data from each record set (table) into Pandas DataFrames for analysis. Use the record set and field `@id`s identified above.

> **Tip:** Use DataFrame column names matching the fields' `@id`s for complete traceability.

In [ ]:
# Extract data from all record sets into dataframes (using @id as column names)
dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"Loaded DataFrame for record set {rs.id} with shape {df.shape}")
        print(f"Columns (field @id): {df.columns.tolist()}\n")
    else:
        print(f"No records found for record set {rs.id}")

# For further demonstration, choose the main tabular record set (typically the largest)
if len(dataframes) > 0:
    # Pick the record set with most records (as main)
    main_rs_id = max(dataframes, key=lambda k: dataframes[k].shape[0])
    print(f"Main record set selected: {main_rs_id}")
    print(dataframes[main_rs_id].head())
else:
    main_rs_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, grouping, using field `@id`s.

- Select a numeric field (e.g., age, interval, or similar, depending on schema). 
- Filter for values above a threshold. 
- Normalize the field (z-score). 
- Group by a key categorical field (if present, e.g., sex, MSI status, anatomical location).

_Replace field and group `@id`s as appropriate; here we use typical medical names_

In [ ]:
# Customize numeric_field_id and group_field_id below after dataframes are loaded/inspected
if main_rs_id is not None:
    df = dataframes[main_rs_id]
    print("Available columns in main data (field @id):\n", df.columns.tolist())
    
    # Example: Look for plausible numeric and grouping fields (adjust @id as necessary)
    possible_numeric = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int] and not df[col].isnull().all()]
    if len(possible_numeric) > 0:
        numeric_field_id = possible_numeric[0]
    else:
        # fallback: find first column name containing e.g. 'age', 'interval', etc.
        numeric_field_id = next((col for col in df.columns if any(x in col.lower() for x in ['age', 'interval', 'days', 'months', 'years'])), df.columns[0])
    
    # For grouping, try 'sex', 'morphology', 'msi', etc.
    group_field_candidates = [col for col in df.columns if any(x in col.lower() for x in ['sex', 'msi', 'location', 'anatomical', 'group', 'site'])]
    group_field_id = group_field_candidates[0] if len(group_field_candidates) > 0 else df.columns[0]
    
    print(f"Using numeric field: {numeric_field_id}")
    print(f"Using group field: {group_field_id}\n")

    # Drop records without valid numeric value
    filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notnull()].copy()
    filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id])

    # Set threshold: 10 (as in template), or use median/mean as a demo
    threshold = 10
    filtered_df_sub = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df_sub.head())

    # Normalize numeric field
    mean_val = filtered_df_sub[numeric_field_id].mean()
    std_val = filtered_df_sub[numeric_field_id].std()
    filtered_df_sub[f"{numeric_field_id}_normalized"] = (filtered_df_sub[numeric_field_id] - mean_val) / std_val if std_val != 0 else 0
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df_sub[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field and show mean of numeric
    if group_field_id in filtered_df_sub.columns:
        grouped_df = filtered_df_sub.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No main record set dataframe to analyze.")

## 5. Visualization
Visualize data distributions or relationships, e.g. distribution of numeric field, or differences grouped by a categorical field. 

- The field `@id`s are annotated in the plots.
- Matplotlib and seaborn are used for quick visualization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id is not None and len(filtered_df_sub) > 0:
    plt.figure(figsize=(8,6))
    sns.histplot(filtered_df_sub[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group
    if group_field_id in filtered_df_sub.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df_sub)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=25)
        plt.show()

## 6. Conclusion
In this notebook, we have:
- Loaded a FAIR² clinical dataset via Croissant schema using `mlcroissant`.
- Explored the available record sets and fields by their `@id` identifiers.
- Loaded the main record set into a DataFrame and performed filtering, normalization, and grouping using relevant fields.
- Visualized data distributions and differences among subgroups.

All operations referenced entities by their `@id` as per Croissant schema best practice.

You can further customize the EDA and feature engineering by changing the numeric and grouping field `@id`s to match specific research needs, or by examining relations among record sets if the dataset provides multiple tables or data types.